# Как воспроизвести Stage 13 V1

1. Откройте PowerShell в корне проекта.
2. Выполните:

   `.\scripts\prepare_stage13_tabfm.ps1`

3. Helper предложит рекомендуемые runtime и HF cache paths. Нажмите Enter, чтобы принять их, либо укажите свои. Для повторяемого запуска без вопросов используйте `-NonInteractive`.
4. При `output guard = PASS` дождитесь строки `ГОТОВО К STAGE 13 SAFE-RUN`. При `output guard = STOP` helper не продолжит запуск: вручную проверьте существующие Stage 13 artifacts и примите отдельное решение; helper их не удаляет и не переименовывает.
5. Откройте этот notebook.
6. Выберите kernel `KOMUS Stage 13 TabFM (Python 3.11)`: он уже содержит выбранные HF cache settings.
7. Для технической проверки оставьте `RUN_FULL_OOF = False`.
8. Выполните Restart Kernel → Run All.

Для явного выбора путей:

```powershell
.\scripts\prepare_stage13_tabfm.ps1 `
    -RuntimeDir "E:\KOMUS\stage13-tabfm-v1" `
    -HfCacheDir "E:\huggingface-cache"
```

Ожидаемый SAFE-RUN acceptance: dataset guards PASS, SHA-256 weights PASS, TabFM model load PASS, real inference preflight PASS, `single-call vs chunked max_abs_diff <= 1e-5` и финальная строка `Safe-run пройден`.

> **Полный OOF:** `RUN_FULL_OOF=True` не включать автоматически. На исходной CPU-машине KOMUS полный OOF не выполнялся из-за неприемлемой вычислительной стоимости; это отдельное осознанное действие после оценки compute.

Final test в Stage 13 не используется.


# Этап 13 V1 — TabFM против `GBDT_mean`

## Исследовательский вопрос

Может ли Google Research TabFM V1 в одном заранее зафиксированном PyTorch/CPU protocol дать существенное улучшение outer-OOF Gini относительно принятого `GBDT_mean`?

Это controlled experiment, а не поиск конфигурации. Неизменны `Data_final.xlsb`, SHA-256 датасета, working sample из 289614 строк, target `DefMark`, identifier `INN`, 47 accepted features в accepted order, StratifiedKFold(3, shuffle=True, random_state=42), fold seeds 43/44/45 и сохранённый Stage 7 OOF baseline. `Q_B1_norm` и `Q_B2_norm` исключены. Final test не читается как model input и не используется.

Используется обычный `TabFMClassifier`, не `TabFMClassifier.ensemble()`. Нет HPO, перебора параметров, stacking/blending, calibration, threshold optimization, class balancing, новых features или fine-tuning. Полный OOF намеренно **не запускается** этим подготовительным notebook: он заблокирован флагом `RUN_FULL_OOF = False`.

### Что проверяем и зачем

Сначала фиксируем environment, provenance, hashes, canonical order и locked protocol. Это исключает молчаливую смену датасета, признаков, baseline или folds до любого model call. Неизменными остаются все accepted Stage 1–12 artifacts.

In [11]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import sys
import threading
import time
import warnings  # pyright: ignore[reportUnusedImport]  # Pylance: import is used in subsequent notebook cells.
from contextlib import contextmanager
from typing import Any
from pathlib import Path

import numpy as np
import pandas as pd  # pyright: ignore[reportUnusedImport]  # Pylance: import is used in subsequent notebook cells.
import torch
from huggingface_hub import constants as hf_constants, snapshot_download  # pyright: ignore[reportUnusedImport]  # Pylance: imports are used in subsequent notebook cells.
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold  # pyright: ignore[reportUnusedImport]  # Pylance: import is used in subsequent notebook cells.
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch  # pyright: ignore[reportUnusedImport]  # Pylance: imports are used in subsequent notebook cells.
from tqdm import tqdm  # pyright: ignore[reportUnusedImport]  # Pylance: import is used in subsequent notebook cells.

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent

DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
GENERATED = ROOT / 'reports' / 'generated'
STAGE1_PATH = GENERATED / 'stage1_baseline_results_V2.json'
STAGE7_PATH = GENERATED / 'stage7_tabm_stacking_results_V1.json'
STAGE7_OOF_PATH = GENERATED / 'stage7_tabm_stacking_oof_V1.npz'
STAGE13_OOF_PATH = GENERATED / 'stage13_tabfm_oof_V1.npz'
STAGE13_RESULT_PATH = GENERATED / 'stage13_tabfm_results_V1.json'

TARGET, IDENTIFIER = 'DefMark', 'INN'
FORBIDDEN = ('Q_B1_norm', 'Q_B2_norm')
EXPECTED_DATASET_SHA = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_SHA = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
EXPECTED_GBDT_GINI = 0.8063993952
EXPECTED_N, EXPECTED_FINAL_N = 289614, 72404
OUTER_SEED, FOLD_SEEDS, THRESHOLD = 42, (43, 44, 45), 0.5

TABFM_SOURCE_COMMIT = 'd8678b6895f1428a468d4cc299c1ff4cf704e726'
TABFM_RELEASE = '1.0.1'
HF_REPO_ID = 'google/tabfm-1.0.0-pytorch'
HF_REVISION = '77cb9cc1b4fd3a9c77fbb9552c218200bb4dab83'
EXPECTED_WEIGHT_SHA = '928cb350becdc77cdb7a9e8c36deda88917bfd14a3091894a2dc516db58a2085'
QUERY_CHUNK_SIZE = 1024
QUERY_EQUIVALENCE_ROWS = 2048
QUERY_EQUIVALENCE_TOLERANCE = 1e-5
HEARTBEAT_SECONDS = 5.0
EXPECTED_PYTHON = (3, 11)
EXPECTED_TORCH_VERSION = '2.12.1+cpu'

TABFM_PARAMS = {
    'n_estimators': 32, 'max_num_rows': 100, 'max_num_features': 500,
    'norm_methods': None, 'feat_shuffle_method': 'random', 'class_shift': True,
    'permute_categorical': False, 'outlier_threshold': 4.0,
    'softmax_temperature': 0.9, 'average_logits': True, 'batch_size': 1,
    'binary_calibration_method': None, 'multiclass_calibration_method': None,
    'n_feature_crosses': 0, 'n_svd_features': 0, 'enable_nnls': False,
}
RUN_FULL_OOF = False  # Safety lock: no full OOF in this notebook state.

def format_stage13_duration(seconds: float) -> str:
    total_seconds = max(0, int(round(seconds)))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def format_bytes(byte_count: float) -> str:
    value = float(byte_count)
    for unit in ('B', 'KB', 'MB', 'GB'):
        if value < 1024.0:
            return f'{value:.2f} {unit}'
        value /= 1024.0
    return f'{value:.2f} TB'

class Stage13LiveStatus:
    """One reusable, exception-safe Jupyter status panel for Stage 13."""
    def __init__(self, heartbeat_seconds: float = HEARTBEAT_SECONDS) -> None:
        self.heartbeat_seconds = heartbeat_seconds
        self._lock = threading.RLock()
        self._display_handle: Any = None
        self._display_available = None
        self._started_at = None
        self._stage_started_at = None
        self._fold_started_at = None
        self._state: dict[str, Any] = {}

    def start_session(self, stage: str) -> None:
        now = time.monotonic()
        with self._lock:
            # Новый run обязан создать output именно в текущей Jupyter-ячейке.
            self._display_handle = None
            self._display_available = None
            self._started_at = now
            self._stage_started_at = now
            self._fold_started_at = None
            self._state = {'stage': stage, 'status': 'RUNNING', 'last_update': time.strftime('%H:%M:%S')}
        self.refresh()

    def _ensure_session(self, stage: str) -> None:
        with self._lock:
            started = self._started_at is not None
        if not started:
            self.start_session(stage)

    def set_stage(self, stage: str, *, fold_id: int | None = None, fold_started_at: float | None = None) -> None:
        self._ensure_session(stage)
        now = time.monotonic()
        with self._lock:
            self._stage_started_at = now
            self._state.update({'stage': stage, 'status': 'RUNNING', 'chunk': None, 'transfer': None, 'inference': None})
            if fold_id is not None:
                self._state['fold_id'] = fold_id
                self._fold_started_at = fold_started_at or now
            elif fold_id is None:
                self._state.pop('fold_id', None)
                self._fold_started_at = None
            self._state['last_update'] = time.strftime('%H:%M:%S')
        self.refresh()

    def set_transfer_progress(self, *, downloaded_bytes: float, total_bytes: float | None, observed_bytes_per_second: float | None) -> None:
        if downloaded_bytes < 0.0 or total_bytes is not None and total_bytes < 0.0:
            raise ValueError('Stage 13 live-status получил недопустимый download progress')
        with self._lock:
            self._state['transfer'] = {
                'downloaded': downloaded_bytes, 'total': total_bytes, 'speed': observed_bytes_per_second,
            }
            self._state['last_update'] = time.strftime('%H:%M:%S')

    def set_inference_progress(
        self, *, fold_processed: int, fold_total: int, stage_processed: int, stage_total: int,
        chunk_id: int, total_chunks: int, fold_inference_elapsed: float, stage_inference_elapsed: float,
    ) -> None:
        if not (0 <= fold_processed <= fold_total and 0 <= stage_processed <= stage_total):
            raise ValueError('Stage 13 live-status получил недопустимый inference progress')
        with self._lock:
            self._state['chunk'] = (chunk_id, total_chunks)
            self._state['inference'] = {
                'fold_processed': fold_processed, 'fold_total': fold_total,
                'stage_processed': stage_processed, 'stage_total': stage_total,
                'fold_inference_elapsed': fold_inference_elapsed,
                'stage_inference_elapsed': stage_inference_elapsed,
            }
            self._state['last_update'] = time.strftime('%H:%M:%S')

    def _panel(self) -> str:
        with self._lock:
            state: dict[str, Any] = dict(self._state)
            started_at, stage_started_at, fold_started_at = self._started_at, self._stage_started_at, self._fold_started_at
        now = time.monotonic()
        stage_elapsed = now - stage_started_at if stage_started_at is not None else 0.0
        lines = ['Stage 13 V1', f"Этап: {state.get('stage', 'ожидание')}", f"Статус: {state.get('status', 'RUNNING')}"]
        if state.get('fold_id') is not None:
            lines.append(f"Fold: {state['fold_id']} / 3")
        if state.get('chunk') is not None:
            chunk_id, total_chunks = state['chunk']
            lines.append(f"Chunk: {chunk_id} / {total_chunks}")
        transfer = state.get('transfer')
        inference = state.get('inference')
        if transfer is not None:
            downloaded, total, speed = transfer['downloaded'], transfer['total'], transfer['speed']
            if total is not None:
                remaining = max(total - downloaded, 0.0)
                lines.extend([
                    f'Загружено: {format_bytes(downloaded)} / {format_bytes(total)}',
                    f'Прогресс: {100.0 * min(downloaded / total, 1.0):.1f}%',
                    f'Осталось: {format_bytes(remaining)}',
                ])
            else:
                remaining = None
                lines.append(f'Загружено: {format_bytes(downloaded)}')
            lines.append(f'Скорость: {format_bytes(speed)}/с' if speed is not None else 'Скорость: недостаточно данных')
            eta = remaining / speed if remaining is not None and speed is not None and speed > 0.0 else None
            lines.append(f'ETA: {format_stage13_duration(eta)}' if eta is not None else 'ETA: неизвестна')
        elif inference is not None:
            def inference_lines(label: str, processed: int, total: int, elapsed: float, eta_label: str) -> list[str]:
                elapsed = max(elapsed, 0.0)
                speed = processed / elapsed if processed > 0 and elapsed > 0.0 else None
                remaining = total - processed
                eta = remaining / speed if speed is not None and speed > 0.0 and remaining > 0 else None
                return [
                    f'{label}:', f'  Обработано: {processed} / {total}',
                    f'  Прогресс: {100.0 * processed / total:.2f}%' if total else '  Прогресс: неизвестен',
                    f'  Осталось: {remaining}',
                    f'  Скорость: {speed:.3f} строк/с' if speed is not None else '  Скорость: недостаточно данных',
                    f'  {eta_label}: {format_stage13_duration(eta)}' if eta is not None else f'  {eta_label}: неизвестна',
                ]
            lines.extend(inference_lines(
                'Fold progress', inference['fold_processed'], inference['fold_total'],
                inference['fold_inference_elapsed'], 'ETA fold',
            ))
            lines.extend(inference_lines(
                'Stage progress', inference['stage_processed'], inference['stage_total'],
                inference['stage_inference_elapsed'], 'ETA stage',
            ))
        else:
            lines.append('ETA: неизвестна')
        lines.append(f'Прошло: {format_stage13_duration(stage_elapsed)}')
        if fold_started_at is not None:
            lines.append(f'Прошло fold: {format_stage13_duration(now - fold_started_at)}')
        if started_at is not None:
            lines.append(f'Прошло всего: {format_stage13_duration(now - started_at)}')
        lines.append(f"Последнее обновление: {state.get('last_update', time.strftime('%H:%M:%S'))}")
        return '\n'.join(lines)

    def refresh(self) -> None:
        with self._lock:
            if self._state:
                self._state['last_update'] = time.strftime('%H:%M:%S')
        panel = self._panel()
        if self._display_available is not False:
            try:
                from IPython.display import Markdown, display
                rendered = Markdown(f'```text\n{panel}\n```')
                if self._display_handle is None:
                    self._display_handle = display(rendered, display_id=True)
                else:
                    self._display_handle.update(rendered)
                self._display_available = True
                return
            except Exception:
                self._display_available = False
        print(panel, flush=True)

    @contextmanager
    def blocking(self, stage: str, *, fold_id: int | None = None, fold_started_at: float | None = None):
        self.set_stage(stage, fold_id=fold_id, fold_started_at=fold_started_at)
        stop_event = threading.Event()
        def heartbeat() -> None:
            while not stop_event.wait(self.heartbeat_seconds):
                self.refresh()
        thread = threading.Thread(target=heartbeat, name='stage13-live-status', daemon=True)
        thread.start()
        try:
            yield
        except BaseException:
            with self._lock:
                self._state['status'] = 'FAILED'
                self._state['last_update'] = time.strftime('%H:%M:%S')
            self.refresh()
            raise
        else:
            with self._lock:
                self._state['status'] = 'PASS'
                self._state['last_update'] = time.strftime('%H:%M:%S')
            self.refresh()
        finally:
            stop_event.set()
            thread.join()

stage13_live = Stage13LiveStatus()

# ОПЕРАТОР: только ручная проверка live-status.
# НЕ запускать через Run All / Run All Above.
# TabFM, dataset и ML эта проверка не запускает.
def demo_stage13_live_status(seconds: int = 30) -> None:
    """Lightweight operator check: only a synthetic blocking wait, no ML or I/O."""
    if not isinstance(seconds, int) or seconds < 0:
        raise ValueError('seconds должен быть неотрицательным целым числом')
    stage13_live.start_session('Демонстрация live-status (synthetic wait)')
    with stage13_live.blocking('Демонстрация live-status (synthetic wait)'):
        time.sleep(seconds)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def sha256_indices(values: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()

def sha256_array(values: np.ndarray) -> str:
    array = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode('utf-8'))
    digest.update(repr(array.shape).encode('utf-8'))
    digest.update(array.tobytes())
    return digest.hexdigest()

def assert_locked_runtime() -> None:
    if sys.version_info[:2] != EXPECTED_PYTHON:
        raise RuntimeError(
            f'STOP: Stage 13 требует Python 3.11.x; получен {platform.python_version()}. ' 
            'Создайте отдельное locked Python 3.11 environment.'
        )
    if torch.__version__ != EXPECTED_TORCH_VERSION:
        raise RuntimeError(
            f'STOP: Stage 13 требует torch {EXPECTED_TORCH_VERSION}; получен {torch.__version__}. ' 
            'Не продолжайте с другим PyTorch runtime.'
        )
    try:
        safetensors_version = importlib.metadata.version('safetensors')
    except importlib.metadata.PackageNotFoundError as error:
        raise RuntimeError(
            'STOP: отсутствует обязательная зависимость среды выполнения safetensors[torch]==0.8.0. ' 
            'Установите точную версию до загрузки модели TabFM.'
        ) from error
    if safetensors_version != '0.8.0':
        raise RuntimeError(
            f'STOP: Stage 13 требует safetensors[torch]==0.8.0; обнаружена версия {safetensors_version}. ' 
            'Установите точную версию до загрузки модели TabFM.'
        )

def assert_stage13_artifacts_absent() -> None:
    existing = [path for path in (STAGE13_OOF_PATH, STAGE13_RESULT_PATH) if path.exists()]
    if existing:
        names = ', '.join(path.name for path in existing)
        raise RuntimeError(
            f'STOP: Stage 13 V1 artifact уже существует: {names}. ' 
            'Автоматический overwrite запрещён; проверьте artifact и удалите его только вручную при явном решении.'
        )

def metrics_at_threshold(target: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    predicted = (probability >= THRESHOLD).astype(np.int8)
    auc = float(roc_auc_score(target, probability))
    return {
        'ROC-AUC': auc, 'Gini': 2.0 * auc - 1.0,
        'PR-AUC': float(average_precision_score(target, probability)),
        'Precision@0.5': float(precision_score(target, predicted, zero_division=0)),
        'Recall@0.5': float(recall_score(target, predicted, zero_division=0)),
        'F1@0.5': float(f1_score(target, predicted, zero_division=0)),
    }

def decision(delta_gini: float, fold_deltas: list[float]) -> str:
    wins = sum(value > 0.0 for value in fold_deltas)
    losses = sum(value < 0.0 for value in fold_deltas)
    if delta_gini >= 0.010 and wins >= 2:
        return 'material_gain'
    if delta_gini <= -0.010 and losses >= 2:
        return 'inferior'
    return 'no_material_benefit'

def json_safe(value):
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    return value

def write_new_npz(path: Path, **arrays: np.ndarray) -> None:
    if path.exists():
        raise RuntimeError(f'STOP: {path.name} уже существует; overwrite запрещён.')
    try:
        with path.open('xb') as stream:
            np.savez_compressed(stream, **arrays)
    except FileExistsError as error:
        raise RuntimeError(f'STOP: {path.name} уже существует; overwrite запрещён.') from error

def write_new_json(path: Path, payload: dict) -> None:
    if path.exists():
        raise RuntimeError(f'STOP: {path.name} уже существует; overwrite запрещён.')
    try:
        with path.open('x', encoding='utf-8') as stream:
            json.dump(json_safe(payload), stream, ensure_ascii=False, indent=2, allow_nan=False)
            stream.write('\n')
    except FileExistsError as error:
        raise RuntimeError(f'STOP: {path.name} уже существует; overwrite запрещён.') from error

def save_stage13_evidence(result: dict, tabfm_oof: np.ndarray) -> dict:
    assert_stage13_artifacts_absent()
    print('Сохранение evidence: записывается Stage 13 OOF artifact без overwrite…')
    write_new_npz(
        STAGE13_OOF_PATH, working_indices=working_indices, target=y_working,
        fold=saved_fold, tabfm_oof=tabfm_oof, gbdt_mean=gbdt_mean,
    )
    expected_arrays = {
        'working_indices': working_indices, 'target': y_working, 'fold': saved_fold,
        'tabfm_oof': tabfm_oof, 'gbdt_mean': gbdt_mean,
    }
    with np.load(STAGE13_OOF_PATH, allow_pickle=False) as saved:
        required = set(expected_arrays)
        assert required.issubset(saved.files), 'STOP: сохранённый Stage 13 OOF artifact неполный'
        assert all(len(saved[name]) == EXPECTED_N for name in required), (
            'STOP: сохранённый Stage 13 OOF artifact имеет неверную длину массивов'
        )
        for name, expected in expected_arrays.items():
            persisted = np.asarray(saved[name])
            assert np.array_equal(persisted, expected), (
                f'STOP: readback Stage 13 OOF artifact: массив {name} не совпадает с in-memory evidence'
            )
            assert sha256_array(persisted) == sha256_array(expected), (
                f'STOP: readback Stage 13 OOF artifact: SHA массива {name} не совпадает'
            )
    oof_sha256 = sha256_file(STAGE13_OOF_PATH)
    saved_result = {
        **result,
        'oof_artifact': {
            'path': str(STAGE13_OOF_PATH.relative_to(ROOT)),
            'sha256': oof_sha256,
        },
    }
    print('Сохранение evidence: записывается Stage 13 result JSON без overwrite…')
    write_new_json(STAGE13_RESULT_PATH, saved_result)
    with STAGE13_RESULT_PATH.open('r', encoding='utf-8') as stream:
        persisted_result = json.load(stream)
    assert persisted_result.get('experiment') == saved_result['experiment'], (
        'STOP: readback Stage 13 result JSON: experiment не совпадает с текущим result'
    )
    assert persisted_result.get('version') == saved_result['version'], (
        'STOP: readback Stage 13 result JSON: version не совпадает с текущим result'
    )
    assert persisted_result.get('decision') == saved_result['decision'], (
        'STOP: readback Stage 13 result JSON: decision не совпадает с текущим result'
    )
    assert persisted_result.get('oof_artifact', {}).get('sha256') == saved_result['oof_artifact']['sha256'], (
        'STOP: readback Stage 13 result JSON: oof_artifact.sha256 не совпадает с текущим result'
    )
    print('Evidence artifacts сохранены и успешно проверены повторным чтением.')
    return saved_result


In [4]:
demo_stage13_live_status(30)

```text
Stage 13 V1
Этап: Preflight fold 1: chunked inference
Статус: PASS
Fold: 1 / 3
Chunk: 2 / 2
Fold progress:
  Обработано: 2048 / 2048
  Прогресс: 100.00%
  Осталось: 0
  Скорость: 0.1 строк/с
  ETA fold: неизвестна
Stage progress:
  Обработано: 2048 / 2048
  Прогресс: 100.00%
  Осталось: 0
  Скорость: 0.1 строк/с
  ETA stage: неизвестна
Прошло: 05:11:22
Прошло fold: 09:45:10
Прошло всего: 09:45:41
Последнее обновление: 00:12:57
```

### Что проверяем и зачем

Проверяем dataset SHA, Stage 1/7 feature identity, canonical working indices, saved fold assignment и saved `GBDT_mean`. Final-test rows проверяются только как complement размера; из них не строится ни `X`, ни `y`, ни prediction. На вход TabFM будет передан ровно один NumPy `float32` matrix из 47 признаков. Собственные imputation, scaling и feature engineering отсутствуют: далее работает только package-native preprocessing `TabFMClassifier`.

In [5]:
assert DATASET.exists() and STAGE1_PATH.exists() and STAGE7_PATH.exists() and STAGE7_OOF_PATH.exists(), 'STOP: отсутствует обязательный input artifact'
assert sha256_file(DATASET) == EXPECTED_DATASET_SHA, 'STOP: SHA-256 Data_final.xlsb не совпадает'
stage1 = json.loads(STAGE1_PATH.read_text(encoding='utf-8'))
stage7 = json.loads(STAGE7_PATH.read_text(encoding='utf-8'))
features = list(stage1['допустимые_признаки'])
assert len(features) == 47 and len(set(features)) == 47, 'STOP: ожидаются ровно 47 уникальных features'
assert not set(FORBIDDEN).intersection(features), 'STOP: Q_B1_norm/Q_B2_norm запрещены как predictors'
assert IDENTIFIER not in features, 'STOP: INN не может быть predictor'
assert features == stage7['raw_features_in_order'], 'STOP: accepted feature order Stage 1/7 не совпадает'
assert stage7['dataset_sha256'] == EXPECTED_DATASET_SHA, 'STOP: dataset SHA Stage 7 не совпадает'
assert stage7['working_index_sha256'] == EXPECTED_WORKING_SHA, 'STOP: working-index SHA Stage 7 не совпадает'
assert stage7['baseline_selection']['B_star'] == 'GBDT_mean', 'STOP: accepted baseline не GBDT_mean'

print('Этап загрузки данных: читается Data_final.xlsb для canonical working sample.')
raw = pd.read_excel(DATASET, engine='pyxlsb')
assert TARGET in raw.columns and IDENTIFIER in raw.columns, 'STOP: target или identifier отсутствует'
assert all(name in raw.columns for name in features), 'STOP: отсутствует accepted feature'
assert len(raw) == EXPECTED_N + EXPECTED_FINAL_N, 'STOP: размер dataset не соответствует locked split'

with np.load(STAGE7_OOF_PATH, allow_pickle=False) as artifact:
    required = {'working_indices', 'target', 'fold', 'gbdt_mean'}
    assert required.issubset(artifact.files), f'STOP: отсутствуют OOF keys: {required - set(artifact.files)}'
    working_indices = np.asarray(artifact['working_indices'], dtype=np.int64)
    y_working = np.asarray(artifact['target'], dtype=np.int8)
    saved_fold = np.asarray(artifact['fold'], dtype=np.int8)
    gbdt_mean = np.asarray(artifact['gbdt_mean'], dtype=np.float64)

assert len(working_indices) == len(y_working) == len(saved_fold) == len(gbdt_mean) == EXPECTED_N, 'STOP: working sample имеет неверную длину'
assert np.unique(working_indices).size == EXPECTED_N, 'STOP: working indices содержат дубликаты'
assert sha256_indices(working_indices) == EXPECTED_WORKING_SHA, 'STOP: canonical working-index SHA не совпадает'
assert np.array_equal(raw.loc[working_indices, TARGET].to_numpy(dtype=np.int8), y_working), 'STOP: target/index mismatch'
assert np.isfinite(gbdt_mean).all() and ((0.0 <= gbdt_mean) & (gbdt_mean <= 1.0)).all(), 'STOP: GBDT_mean содержит невалидные probabilities'
assert abs(metrics_at_threshold(y_working, gbdt_mean)['Gini'] - EXPECTED_GBDT_GINI) <= 1e-9, 'STOP: GBDT_mean Gini не совпадает с accepted evidence'

splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED)
outer_splits = list(splitter.split(np.zeros(EXPECTED_N, dtype=np.int8), y_working))
expected_fold = np.zeros(EXPECTED_N, dtype=np.int8)
for fold_id, (_, valid_pos) in enumerate(outer_splits, start=1):
    expected_fold[valid_pos] = fold_id
assert np.array_equal(saved_fold, expected_fold), 'STOP: saved fold assignment не соответствует accepted outer CV'
assert np.setdiff1d(np.arange(len(raw)), working_indices).size == EXPECTED_FINAL_N, 'STOP: final-test complement не соответствует locked split'

# Explicit final-test gate: only canonical working rows are materialized below.
FINAL_TEST_USED = False
assert FINAL_TEST_USED is False, 'STOP: final test запрещён в Stage 13 V1'
X_working = raw.loc[working_indices, features].to_numpy(dtype=np.float32, copy=True)
assert X_working.shape == (EXPECTED_N, 47) and X_working.dtype == np.float32, 'STOP: TabFM input должен быть [289614, 47] NumPy float32'
assert np.isfinite(X_working).all(), 'STOP: unexpected NaN/Inf в accepted TabFM input; собственная imputation запрещена'
assert np.isfinite(y_working).all() and set(np.unique(y_working)) == {0, 1}, 'STOP: target должен быть бинарным и finite'
print(f'Preflight пройден: working n={EXPECTED_N}; features=47; GBDT Gini={EXPECTED_GBDT_GINI:.10f}; final test заблокирован.')


Этап загрузки данных: читается Data_final.xlsb для canonical working sample.
Preflight пройден: working n=289614; features=47; GBDT Gini=0.8063993952; final test заблокирован.


### Что проверяем и зачем

До model load hard guard требует именно Python 3.11.x и `torch==2.12.1+cpu`. Затем убеждаемся, что установлен source commit TabFM 1.0.1, скачивается фиксированный HF revision и `classification/model.safetensors` имеет expected SHA-256. PyTorch-модель загружается на CPU в `bfloat16`; dtype и device затем входят в воспроизводимую metadata. Вывод ограничен progress-сообщениями, без перечисления локальных файлов.

In [6]:
def installed_tabfm_commit() -> str | None:
    direct_url = importlib.metadata.distribution('tabfm').read_text('direct_url.json')
    if not direct_url:
        return None
    payload = json.loads(direct_url)
    return payload.get('vcs_info', {}).get('commit_id')

assert importlib.metadata.version('tabfm') == TABFM_RELEASE, 'STOP: tabfm release должен быть 1.0.1'
assert installed_tabfm_commit() == TABFM_SOURCE_COMMIT, 'STOP: установлен не locked google-research/tabfm commit'

def format_load_duration(seconds: float) -> str:
    total_seconds = max(0, int(round(seconds)))
    minutes, seconds = divmod(total_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

class HumanDownloadProgress(tqdm):
    """Silent Hugging Face tqdm adapter with an aggregated Russian heartbeat."""
    _lock = threading.Lock()
    _downloaded_bytes = 0.0
    _known_total_bytes = 0.0
    _last_report_at = 0.0
    _started_at = 0.0
    _last_sample_bytes = 0.0
    _last_sample_at = 0.0

    @classmethod
    def reset_session(cls) -> None:
        with cls._lock:
            cls._downloaded_bytes = 0.0
            cls._known_total_bytes = 0.0
            cls._last_report_at = time.monotonic()
            cls._started_at = cls._last_report_at
            cls._last_sample_bytes = 0.0
            cls._last_sample_at = cls._started_at

    def __init__(self, *args, **kwargs):
        self._is_byte_progress = kwargs.get('unit') == 'B'
        initial = float(kwargs.get('initial') or 0.0)
        total = kwargs.get('total')
        self._initial_bytes = initial if self._is_byte_progress else 0.0
        self._total_bytes = float(total) if self._is_byte_progress and total else 0.0
        kwargs['disable'] = True
        super().__init__(*args, **kwargs)
        if self._is_byte_progress:
            with type(self)._lock:
                type(self)._downloaded_bytes += self._initial_bytes
                type(self)._known_total_bytes += self._total_bytes

    def _sync_total_from_tqdm(self) -> None:
        if not self._is_byte_progress:
            return
        total = getattr(self, 'total', None)
        if total is None:
            return
        total = float(total)
        with type(self)._lock:
            type(self)._known_total_bytes += total - self._total_bytes
            self._total_bytes = total

    def refresh(self, *args, **kwargs):
        self._sync_total_from_tqdm()
        return super().refresh(*args, **kwargs)

    def update(self, n=1):
        if self._is_byte_progress and n:
            with type(self)._lock:
                type(self)._downloaded_bytes += float(n)
        result = super().update(n)
        type(self).report_if_due()
        return result

    @classmethod
    def report_if_due(cls, force: bool = False) -> None:
        with cls._lock:
            now = time.monotonic()
            if not force and now - cls._last_report_at < HEARTBEAT_SECONDS:
                return
            cls._last_report_at = now
            downloaded = cls._downloaded_bytes
            total = cls._known_total_bytes
            elapsed = now - cls._started_at
            sample_elapsed = now - cls._last_sample_at
            sample_bytes = downloaded - cls._last_sample_bytes
            cls._last_sample_at = now
            cls._last_sample_bytes = downloaded
        speed = sample_bytes / sample_elapsed if sample_elapsed > 0.0 and sample_bytes > 0.0 else None
        stage13_live.set_transfer_progress(
            downloaded_bytes=downloaded, total_bytes=total if total > 0.0 else None,
            observed_bytes_per_second=speed,
        )

def load_locked_tabfm_model():
    cache_dir = Path(hf_constants.HF_HUB_CACHE)
    started_at = time.monotonic()
    print(
        'Stage 13 V1 — загрузка pretrained TabFM\n'
        f'Cache: {cache_dir}\n'
        f'Модель: {HF_REPO_ID}\n'
        f'Revision: {HF_REVISION}\n'
        'Artifact: classification/model.safetensors\n'
        'Общий размер: будет показан только после получения надёжной metadata\n'
        'Статус: проверяем локальный cache / начинаем загрузку'
    )
    HumanDownloadProgress.reset_session()
    with stage13_live.blocking('TabFM: загрузка pinned weights'):
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore', message='.*unauthenticated.*', category=UserWarning,
                module=r'huggingface_hub.*',
            )
            weights_root = Path(snapshot_download(
                repo_id=HF_REPO_ID, revision=HF_REVISION,
                allow_patterns=['classification/*'], tqdm_class=HumanDownloadProgress,
            ))
            HumanDownloadProgress.report_if_due(force=True)
    weight_path = weights_root / 'classification' / 'model.safetensors'
    assert weight_path.exists(), 'STOP: classification/model.safetensors не найден в locked HF snapshot'
    download_elapsed = time.monotonic() - started_at
    print(
        'Загрузка TabFM weights завершена.\n'
        f'Размер: {format_bytes(weight_path.stat().st_size)}\n'
        f'Время загрузки: {format_load_duration(download_elapsed)}\n'
        'Проверяется SHA-256…'
    )
    assert sha256_file(weight_path) == EXPECTED_WEIGHT_SHA, 'STOP: SHA-256 classification/model.safetensors не совпадает'
    print('SHA-256 weights: PASS')
    print(
        '\nЭтап                    Статус       Объём / время\n'
        'Проверка runtime        OK           before model load\n'
        f'Pretrained weights      READY        {format_bytes(weight_path.stat().st_size)}\n'
        f'Cache                   {cache_dir}\n'
        f'Загрузка                завершена    {format_load_duration(download_elapsed)}'
    )
    print('Инициализация PyTorch/CPU…')
    with stage13_live.blocking('TabFM: model load PyTorch/CPU'):
        model = tabfm_v1_0_0_pytorch.load(
            model_type='classification', checkpoint_path=str(weights_root),
            device='cpu', dtype=torch.bfloat16, use_cache=False,
        )
    device = str(next(model.parameters()).device)
    dtype = str(next(model.parameters()).dtype)
    assert device == 'cpu' and dtype == 'torch.bfloat16', 'STOP: locked PyTorch CPU/bfloat16 model не получен'
    return model, {'device': device, 'dtype': dtype, 'weight_sha256': EXPECTED_WEIGHT_SHA}

def package_versions() -> dict[str, str]:
    return {name: importlib.metadata.version(name) for name in (
        'tabfm', 'torch', 'numpy', 'pandas', 'scikit-learn', 'huggingface-hub', 'pyxlsb'
    )}


### Что проверяем и зачем

Эти функции формируют fresh classifier для каждого outer fold, сохраняют exact sampled-row и feature-shuffle patterns вместе с hashes и выполняют inference в query chunks по 1024 строки. `predict_positive_chunked` не получает labels. Русский monitor выводится при старте и завершении fold, а между ними — только time-based примерно раз в 45 секунд. Проценты считаются по фактически обработанным validation rows; ETA появляется лишь после нескольких chunks или 60 секунд измеренного времени и вычисляется только из наблюдаемой скорости.

In [7]:
def make_classifier(model, fold_seed: int) -> TabFMClassifier:
    # Direct constructor is mandatory: do not call TabFMClassifier.ensemble().
    return TabFMClassifier(model=model, random_state=fold_seed, **TABFM_PARAMS)

def hash_json(value) -> str:
    encoded = json.dumps(json_safe(value), ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()

def fold_pattern_metadata(classifier: TabFMClassifier, train_working_indices: np.ndarray) -> dict:
    generator = classifier.ensemble_generator_
    feature_patterns = {
        str(method): [np.asarray(pattern, dtype=np.int64).tolist() for pattern in patterns]
        for method, patterns in generator.feature_shuffle_patterns_.items()
    }
    row_patterns = {}
    for method, patterns in generator.row_subsample_patterns_.items():
        row_patterns[str(method)] = [
            None if pattern is None else train_working_indices[np.asarray(pattern, dtype=np.int64)].tolist()
            for pattern in patterns
        ]
    return {
        'feature_shuffle_patterns': feature_patterns,
        'feature_shuffle_patterns_sha256': hash_json(feature_patterns),
        'sampled_working_index_patterns': row_patterns,
        'sampled_working_index_patterns_sha256': hash_json(row_patterns),
    }

def positive_probability(classifier: TabFMClassifier, query: np.ndarray) -> np.ndarray:
    assert query.dtype == np.float32 and query.ndim == 2 and query.shape[1] == 47, 'STOP: query должен быть float32 с 47 features'
    probability = np.asarray(classifier.predict_proba(query), dtype=np.float64)
    assert np.array_equal(classifier.classes_, np.array([0, 1])), 'STOP: TabFM classes должны быть [0, 1]'
    result = probability[:, 1]
    assert np.isfinite(result).all() and ((0.0 <= result) & (result <= 1.0)).all(), 'STOP: TabFM выдал NaN/Inf или probability вне [0,1]'
    return result

def predict_positive_chunked(
    classifier: TabFMClassifier, query: np.ndarray, *, fold_id: int,
    completed_stage_rows: int, completed_stage_inference_seconds: float, stage_total: int,
    show_progress: bool = True, inference_timing: dict[str, float] | None = None,
) -> np.ndarray:
    inference_started_at = time.monotonic()
    output = np.empty(len(query), dtype=np.float64)
    total_chunks = (len(query) + QUERY_CHUNK_SIZE - 1) // QUERY_CHUNK_SIZE
    for chunk_id, start in enumerate(range(0, len(query), QUERY_CHUNK_SIZE), start=1):
        stop = min(start + QUERY_CHUNK_SIZE, len(query))
        output[start:stop] = positive_probability(classifier, query[start:stop])
        processed_fold_rows = stop
        if show_progress:
            current_inference_elapsed = time.monotonic() - inference_started_at
            stage13_live.set_inference_progress(
                fold_processed=processed_fold_rows, fold_total=len(query),
                stage_processed=completed_stage_rows + processed_fold_rows, stage_total=stage_total,
                chunk_id=chunk_id, total_chunks=total_chunks,
                fold_inference_elapsed=current_inference_elapsed,
                stage_inference_elapsed=completed_stage_inference_seconds + current_inference_elapsed,
            )
    inference_elapsed = time.monotonic() - inference_started_at
    if inference_timing is not None:
        inference_timing['elapsed_seconds'] = inference_elapsed
    assert np.isfinite(output).all(), 'STOP: chunked inference содержит NaN/Inf'
    return output

def assert_single_call_equals_chunked(model) -> dict:
    train_pos, valid_pos = outer_splits[0]
    query_pos = valid_pos[:min(QUERY_EQUIVALENCE_ROWS, len(valid_pos))]
    print('Preflight inference: готовится fold 1 context для single-call/chunked проверки…')
    classifier = make_classifier(model, FOLD_SEEDS[0])
    preflight_started = time.monotonic()
    with stage13_live.blocking('Preflight fold 1: fit-context', fold_id=1, fold_started_at=preflight_started):
        classifier.fit(X_working[train_pos], y_working[train_pos])
    # No validation labels are used here; only query features are compared.
    with stage13_live.blocking('Preflight fold 1: single-call inference', fold_id=1, fold_started_at=preflight_started):
        single = positive_probability(classifier, X_working[query_pos])
    with stage13_live.blocking('Preflight fold 1: chunked inference', fold_id=1, fold_started_at=preflight_started):
        chunked = predict_positive_chunked(
            classifier, X_working[query_pos], fold_id=1, completed_stage_rows=0,
            completed_stage_inference_seconds=0.0, stage_total=len(query_pos), show_progress=True,
        )
    max_abs_diff = float(np.max(np.abs(single - chunked)))
    assert max_abs_diff <= QUERY_EQUIVALENCE_TOLERANCE, (
        f'STOP: single-call vs chunked max_abs_diff={max_abs_diff:.8g} > {QUERY_EQUIVALENCE_TOLERANCE}'
    )
    print(f'Preflight inference пройден: max_abs_diff={max_abs_diff:.8g} <= {QUERY_EQUIVALENCE_TOLERANCE}')
    return {'rows': int(len(query_pos)), 'max_abs_diff': max_abs_diff, 'tolerance': QUERY_EQUIVALENCE_TOLERANCE}


### Что проверяем и зачем

Следующая функция — единственный full-run entrypoint. Перед model load она проверяет runtime и отсутствие V1 artifacts, а перед OOF обязательно проводит single-call/chunked preflight и останавливается при `max_abs_diff > 1e-5`. Для каждого fold создаётся fresh `TabFMClassifier`, `fit` получает только outer-train, а validation labels используются только после возврата и размещения fold predictions в canonical order. После успешного run OOF и полный result сохраняются в новые V1 artifacts; любой существующий artifact вызывает STOP без overwrite. Сохранённые NPZ и JSON немедленно открываются повторно: arrays проверяются по equality и deterministic SHA, а ключевые JSON поля — против текущего result.

In [8]:
# ОПЕРАТОР: при RUN_FULL_OOF=False выполняется только SAFE-RUN.
# Полный 3-fold OOF запускается только после ручной установки RUN_FULL_OOF=True.
def run_locked_tabfm_oof() -> dict:
    if not RUN_FULL_OOF:
        raise RuntimeError(
            'STOP: full OOF заблокирован. Для осознанного запуска установите RUN_FULL_OOF = True и выполните эту ячейку заново.')
    assert_locked_runtime()
    assert_stage13_artifacts_absent()
    stage13_live.start_session('Full OOF: подготовка Stage 13')
    started = time.monotonic()
    print('Stage 13 V1: загрузка locked TabFM PyTorch/CPU model…')
    model, device_metadata = load_locked_tabfm_model()
    equivalence = assert_single_call_equals_chunked(model)
    tabfm_oof = np.full(EXPECTED_N, np.nan, dtype=np.float64)
    fold_rows, reproducibility_folds = [], []
    completed_stage_rows = 0
    completed_stage_inference_seconds = 0.0

    for fold_id, ((train_pos, valid_pos), fold_seed) in enumerate(zip(outer_splits, FOLD_SEEDS), start=1):
        fold_started = time.monotonic()
        print(
            f'Fold {fold_id}/3: подготовка context; seed={fold_seed}; train={len(train_pos)}; valid={len(valid_pos)}')
        classifier = make_classifier(model, fold_seed)
        # The only labels passed to fit are outer-train labels.
        with stage13_live.blocking(f'Fold {fold_id}/3: fit-context', fold_id=fold_id, fold_started_at=fold_started):
            classifier.fit(X_working[train_pos], y_working[train_pos])
        fold_metadata = fold_pattern_metadata(
            classifier, working_indices[train_pos])
        print(f'Fold {fold_id}/3: context готов; начинается chunked inference…')
        fold_inference_timing = {}
        with stage13_live.blocking(f'Fold {fold_id}/3: chunked inference', fold_id=fold_id, fold_started_at=fold_started):
            fold_probability = predict_positive_chunked(
                classifier, X_working[valid_pos], fold_id=fold_id,
                completed_stage_rows=completed_stage_rows,
                completed_stage_inference_seconds=completed_stage_inference_seconds,
                stage_total=EXPECTED_N, inference_timing=fold_inference_timing,
            )
        # Return predictions to canonical working-index order before touching validation labels.
        tabfm_oof[valid_pos] = fold_probability
        assert np.isfinite(tabfm_oof[valid_pos]).all(
        ), 'STOP: fold predictions не возвращены в canonical order'
        with stage13_live.blocking(f'Fold {fold_id}/3: metrics', fold_id=fold_id, fold_started_at=fold_started):
            fold_tabfm = metrics_at_threshold(
                y_working[valid_pos], tabfm_oof[valid_pos])
            fold_gbdt = metrics_at_threshold(
                y_working[valid_pos], gbdt_mean[valid_pos])
        fold_delta = fold_tabfm['Gini'] - fold_gbdt['Gini']
        elapsed = time.monotonic() - fold_started
        fold_rows.append({'fold': fold_id, 'tabfm': fold_tabfm, 'GBDT_mean': fold_gbdt,
                         'delta_gini': fold_delta, 'runtime_seconds': elapsed})
        reproducibility_folds.append(
            {'fold': fold_id, 'seed': fold_seed, **fold_metadata})
        completed_stage_rows += len(valid_pos)
        completed_stage_inference_seconds += fold_inference_timing['elapsed_seconds']
        print(
            f'Fold {fold_id}/3 завершён: ΔGini={fold_delta:+.6f}; elapsed={elapsed:.1f}s')

    assert np.isfinite(tabfm_oof).all(), 'STOP: OOF заполнен не полностью'
    tabfm_metrics = metrics_at_threshold(y_working, tabfm_oof)
    baseline_metrics = metrics_at_threshold(y_working, gbdt_mean)
    deltas = {name: tabfm_metrics[name] -
              baseline_metrics[name] for name in tabfm_metrics}
    fold_deltas = [row['delta_gini'] for row in fold_rows]
    result = {
        'experiment': 'Stage 13', 'version': 'V1', 'status': 'completed',
        'dataset_sha256': EXPECTED_DATASET_SHA, 'working_index_sha256': EXPECTED_WORKING_SHA,
        'target': TARGET, 'identifier': IDENTIFIER, 'features_in_accepted_order': features,
        'forbidden_features': list(FORBIDDEN), 'final_test_used': False,
        'baseline': {'name': 'GBDT_mean', 'oof_source': str(STAGE7_OOF_PATH.relative_to(ROOT)), 'retrained': False},
        'tabfm': {
            'source_commit': TABFM_SOURCE_COMMIT, 'release': TABFM_RELEASE,
            'hf_repo_id': HF_REPO_ID, 'hf_revision': HF_REVISION,
            **device_metadata, 'constructor': 'TabFMClassifier', 'ensemble_preset_used': False,
            'locked_parameters': TABFM_PARAMS, 'query_chunk_size': QUERY_CHUNK_SIZE,
            'single_call_vs_chunked': equivalence,
        },
        'package_versions': package_versions(), 'python': platform.python_version(),
        'outer_cv': {'type': 'StratifiedKFold', 'n_splits': 3, 'shuffle': True, 'random_state': OUTER_SEED},
        'fold_reproducibility': reproducibility_folds, 'fold_metrics': fold_rows,
        'tabfm_oof_metrics': tabfm_metrics, 'gbdt_mean_oof_metrics': baseline_metrics,
        'delta_tabfm_minus_gbdt_mean': deltas, 'decision': decision(deltas['Gini'], fold_deltas),
        'runtime_seconds': time.monotonic() - started,
        'limitations': [
            'Random outer CV не доказывает temporal stability.',
            'Три folds не являются claim о statistical significance.',
            'Precision/Recall/F1 at 0.5 — только диагностика.',
            'Final test не использован.',
            'Проверена одна locked TabFM V1 configuration без KOMUS-specific HPO.',
        ],
    }
    with stage13_live.blocking('Сохранение Stage 13 evidence artifacts'):
        saved_result = save_stage13_evidence(result, tabfm_oof)
    print(
        f"Stage 13 V1 завершён: decision={saved_result['decision']}; OOF ΔGini={deltas['Gini']:+.6f}; elapsed={saved_result['runtime_seconds']:.1f}s")
    return saved_result


stage13_result = None
if RUN_FULL_OOF:
    stage13_result = run_locked_tabfm_oof()
else:
    stage13_live.start_session('Safe-run: подготовка Stage 13')
    print('Safe-run Stage 13: проверяем locked runtime и TabFM model preflight без полного OOF.')
    assert_locked_runtime()
    assert_stage13_artifacts_absent()

    safe_model, _ = load_locked_tabfm_model()
    safe_preflight = assert_single_call_equals_chunked(safe_model)

    print(
        'Safe-run пройден: runtime, pinned weights и inference preflight проверены; '
        'RUN_FULL_OOF=False, полный 3-fold OOF не запущен.'
    )

Safe-run Stage 13: проверяем locked runtime и TabFM model preflight без полного OOF.
Stage 13 V1 — загрузка pretrained TabFM
Cache: D:\huggingface-cache\hub
Модель: google/tabfm-1.0.0-pytorch
Revision: 77cb9cc1b4fd3a9c77fbb9552c218200bb4dab83
Artifact: classification/model.safetensors
Общий размер: будет показан только после получения надёжной metadata
Статус: проверяем локальный cache / начинаем загрузку
Загрузка TabFM weights завершена.
Размер: 6.11 GB
Время загрузки: 00:00:00
Проверяется SHA-256…
SHA-256 weights: PASS

Этап                    Статус       Объём / время
Проверка runtime        OK           before model load
Pretrained weights      READY        6.11 GB
Cache                   D:\huggingface-cache\hub
Загрузка                завершена    00:00:00
Инициализация PyTorch/CPU…
Loading weights from local directory
Preflight inference: готовится fold 1 context для single-call/chunked проверки…
Preflight inference пройден: max_abs_diff=0 <= 1e-05
Safe-run пройден: runtime, pi

## Итоговый вывод

Ячейка ниже автоматически формирует результат в двух состояниях. При `RUN_FULL_OOF=False` она явно сообщает, что Stage 13 results ещё нет. После будущего успешного run она выводит только реальные значения из сохранённого `stage13_result`, без заранее вписанных чисел.

In [ ]:
def render_stage13_conclusion(result: dict | None) -> str:
    if result is None:
        return '''## Результат исследования

### FACTS

Notebook подготовлен; `RUN_FULL_OOF=False`, полный TabFM OOF не выполнен и результатов Stage 13 ещё нет. `GBDT_mean` остаётся сохранённым Stage 7 OOF comparator.

### INTERPRETATION

До полного locked run нельзя делать вывод о качестве TabFM.

### LIMITATIONS

Final test закрыт; random outer CV не доказывает temporal stability.

### NEXT STEP

В отдельном Python 3.11 environment выполнить preflight и только после review осознанно установить `RUN_FULL_OOF=True`.'''

    tabfm = result['tabfm_oof_metrics']
    baseline = result['gbdt_mean_oof_metrics']
    delta = result['delta_tabfm_minus_gbdt_mean']
    fold_deltas = ', '.join(f"{row['delta_gini']:+.6f}" for row in result['fold_metrics'])
    return f'''## Результат исследования

### FACTS

- TabFM OOF Gini: **{tabfm['Gini']:.6f}**.
- GBDT_mean OOF Gini: **{baseline['Gini']:.6f}**.
- ΔGini TabFM − GBDT_mean: **{delta['Gini']:+.6f}**.
- Fold ΔGini: **{fold_deltas}**.
- TabFM PR-AUC: **{tabfm['PR-AUC']:.6f}**; Precision@0.5: **{tabfm['Precision@0.5']:.6f}**; Recall@0.5: **{tabfm['Recall@0.5']:.6f}**; F1@0.5: **{tabfm['F1@0.5']:.6f}**.
- Runtime: **{result['runtime_seconds']:.1f} сек**.
- Decision: **{result['decision']}**.
- Evidence: `{result['oof_artifact']['path']}`; SHA-256 `{result['oof_artifact']['sha256']}`.

### INTERPRETATION

Decision вычислен заранее locked rule по полному OOF ΔGini и направлению fold deltas; диагностические метрики при 0.5 его не изменяют.

### LIMITATIONS

Random outer CV не доказывает temporal stability или statistical significance; final test не использован.

### NEXT STEP

Сохранённые V1 evidence artifacts не перезаписывать; интерпретировать результат только в рамках locked Stage 13 protocol.'''

print(render_stage13_conclusion(stage13_result))


## Результат исследования

### FACTS

Notebook подготовлен; `RUN_FULL_OOF=False`, полный TabFM OOF не выполнен и результатов Stage 13 ещё нет. `GBDT_mean` остаётся сохранённым Stage 7 OOF comparator.

### INTERPRETATION

До полного locked run нельзя делать вывод о качестве TabFM.

### LIMITATIONS

Final test закрыт; random outer CV не доказывает temporal stability.

### NEXT STEP

В отдельном Python 3.11 environment выполнить preflight и только после review осознанно установить `RUN_FULL_OOF=True`.


## Итог Stage 13 V1 — TabFM

### ФАКТЫ

Stage 13 V1 успешно прошёл SAFE-RUN для зафиксированного TabFM pipeline.

Подтверждено:

- locked TabFM model успешно загружена на CPU;
- SHA-256 checkpoint weights совпал с ожидаемым значением;
- реальный inference preflight успешно выполнен;
- сравнение single-call и chunked inference дало `max_abs_diff = 0`, что удовлетворяет проверке `<= 1e-5`;
- `RUN_FULL_OOF=False`;
- полный 3-fold OOF не запускался;
- OOF-метрики качества TabFM — `Gini`, `PR-AUC`, `Recall`, `Precision`, `F1` и другие — отсутствуют;
- final test не использовался.

SAFE-RUN также показал, что inference TabFM на текущей CPU-среде практически очень медленный.

Общий elapsed SAFE-RUN нельзя использовать как точную оценку runtime полного OOF, поскольку в notebook были периоды sleep/idle. Однако наблюдаемая скорость inference указывает, что полный OOF на текущем CPU потребовал бы многодневного, потенциально многонедельного выполнения.

### ИНТЕРПРЕТАЦИЯ И ОГРАНИЧЕНИЯ

Stage 13 V1 подтвердил **техническую исполнимость зафиксированного TabFM pipeline**: checkpoint корректно загружается, настоящий inference выполняется, а chunked и single-call режимы дают согласованный результат.

При этом Stage 13 **не ответил на исходный вопрос качества**:

> улучшает ли TabFM полный OOF Gini относительно принятого `GBDT_mean`?

Причина отсутствия ответа — не доказанная проблема качества модели, а неприемлемая вычислительная стоимость полного controlled OOF на текущей CPU-среде.

SAFE-RUN нельзя использовать как proxy качества модели.

В частности:

- результат preflight на небольшом числе строк нельзя экстраполировать на весь working sample;
- по SAFE-RUN нельзя делать выводы о полном OOF Gini, PR-AUC, Recall или других метриках;
- TabFM нельзя сравнивать с `GBDT_mean` по качеству без полного OOF;
- общий elapsed SAFE-RUN частично включает sleep/idle и поэтому не даёт точного runtime будущего full OOF.

Тем не менее наблюдаемой CPU throughput достаточно для operational decision:

**продолжать полный OOF в текущей вычислительной среде сейчас нерационально.**

Это ограничение compute setup, а не evidence того, что TabFM является плохой или хорошей моделью.

### РЕШЕНИЕ И СЛЕДУЮЩИЙ ШАГ

**Stage 13 V1 остановить на SAFE-RUN. Полный OOF на текущей CPU-среде не выполнять.**

Статус:

`STOPPED_BY_COMPUTE_COST`

То есть Stage 13 V1 технически валидирован, но остановлен из-за неприемлемой вычислительной стоимости полного OOF в текущей среде.

Дальнейшие действия:

- сохранить notebook и полученное SAFE-RUN evidence;
- оставить `RUN_FULL_OOF=False`;
- не создавать Stage 13 OOF/results artifacts с отсутствующими или выдуманными метриками;
- не использовать final test;
- не менять принятый `GBDT_mean` baseline;
- не делать выводов, что TabFM лучше или хуже `GBDT_mean`;
- вернуться к research synthesis, evidence package и подготовке presentation / defence;
- не открывать новый model search без новой содержательной гипотезы, нового источника информации или нового compute setup.

TabFM можно переоткрыть позже только как **новый controlled experiment** с заранее зафиксированным protocol, если появится существенно более подходящая вычислительная среда.